In [1]:
import pandas as pd
import numpy as np

# Configuration [cite: 494, 495]
STUDENT_ID = 20251234 
df = pd.read_csv('most-popular-programming-languages-2004-2024.csv')
df['Month'] = pd.to_datetime(df['Month'])

# Assign Language [cite: 496]
languages = sorted([col for col in df.columns if col != 'Month'])
assigned_lang = languages[(STUDENT_ID % 1000) % len(languages)]

# Prep Data
lc_data = df[['Month', assigned_lang]].copy()
lc_data.rename(columns={assigned_lang: 'Popularity'}, inplace=True)

# Calculations [cite: 497, 498]
lc_data['Growth_Rate'] = lc_data['Popularity'].pct_change() * 100
lc_data['Moving_Avg'] = lc_data['Popularity'].rolling(window=6).mean()
lc_data['Moving_STD'] = lc_data['Popularity'].rolling(window=6).std()

# Statistical thresholds [cite: 499]
mean_growth = lc_data['Growth_Rate'].mean()
std_growth = lc_data['Growth_Rate'].std()

# Lifecycle Classification Rules [cite: 500, 501]
conditions = [
    (lc_data['Growth_Rate'] > mean_growth),                             # Growth
    (lc_data['Growth_Rate'] > 0) & (lc_data['Growth_Rate'] <= mean_growth), # Introduction
    (lc_data['Growth_Rate'].abs() <= 1.0),                             # Maturity
    (lc_data['Growth_Rate'] < 0) & (lc_data['Growth_Rate'] < -std_growth) # Decline
]
stages = ['Growth', 'Introduction', 'Maturity', 'Decline']
lc_data['Lifecycle_Phase'] = np.select(conditions, stages, default='Maturity')

# Summary Output [cite: 502, 503]
print(f"Analysis for: {assigned_lang}")
print(lc_data[['Month', 'Popularity', 'Growth_Rate', 'Lifecycle_Phase']].tail())
print("\nPhase Distribution:")
print(lc_data['Lifecycle_Phase'].value_counts(normalize=True) * 100)

Analysis for: Matlab Worldwide(%)
         Month  Popularity  Growth_Rate Lifecycle_Phase
244 2024-05-01          69    -1.428571        Maturity
245 2024-06-01          67    -2.898551        Maturity
246 2024-07-01          65    -2.985075        Maturity
247 2024-08-01          65     0.000000        Maturity
248 2024-09-01          71     9.230769          Growth

Phase Distribution:
Lifecycle_Phase
Maturity    43.775100
Growth      42.168675
Decline     14.056225
Name: proportion, dtype: float64
